In [1]:
import pandas as pd
import json
from sqlalchemy import create_engine

In [4]:
engine = create_engine('sqlite:///yelp.db')

In [5]:
#Function to load JSON data into SQLite database in chunks
import json
import pandas as pd

def load_json_to_sqlite(json_file, table_name, engine, chunksize=10000):

    first_chunk = True

    for chunk in pd.read_json(
        json_file,
        lines=True,
        chunksize=chunksize
    ):

        # Convert dict/list columns to text
        for col in chunk.columns:
            if chunk[col].apply(lambda x: isinstance(x, (dict, list))).any():
                chunk[col] = chunk[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
                )

        chunk.to_sql(
            table_name,
            con=engine,
            if_exists='replace' if first_chunk else 'append',
            index=False
        )

        first_chunk = False

    print(f"{table_name} loaded successfully")

In [6]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS business"))

In [8]:
# Load business data into SQLite database
load_json_to_sqlite(
    '../data/yelp_academic_dataset_business.json',
    'business',
    engine
)

business loaded successfully


In [9]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM business", engine)

,total_rows
0,150346


In [10]:
pd.read_sql("SELECT * FROM business LIMIT 5", engine)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,"{""ByAppointmentOnly"": ""True""}","Doctors, Traditional Chinese Medicine, Naturop...",NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,"{""BusinessAcceptsCreditCards"": ""True""}","Shipping Centers, Local Services, Notaries, Ma...","{""Monday"": ""0:0-0:0"", ""Tuesday"": ""8:0-18:30"", ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{""BikeParking"": ""True"", ""BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{""Monday"": ""8:0-22:0"", ""Tuesday"": ""8:0-22:0"", ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{""RestaurantsDelivery"": ""False"", ""OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{""Monday"": ""7:0-20:0"", ""Tuesday"": ""7:0-20:0"", ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{""BusinessAcceptsCreditCards"": ""True"", ""Wheelc...","Brewpubs, Breweries, Food","{""Wednesday"": ""14:0-22:0"", ""Thursday"": ""16:0-2..."


In [11]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS checkin"))

In [13]:
# Load checkin data into SQLite database
load_json_to_sqlite(
    '../data/yelp_academic_dataset_checkin.json',
    'checkin',
    engine
)

checkin loaded successfully


In [15]:
# Verify tip data if already loaded
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tip"))
# Load tip data into SQLite database
load_json_to_sqlite(
    '../data/yelp_academic_dataset_tip.json',
    'tip',
    engine
)

tip loaded successfully


In [16]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table'
"""

pd.read_sql(query, engine)

,name
0,business
1,checkin
2,tip


In [17]:
# Check the schema of the business table
pd.read_sql("PRAGMA table_info(business);", engine)

,cid,name,type,notnull,dflt_value,pk
0,0,business_id,TEXT,0,None,0
1,1,name,TEXT,0,None,0
2,2,address,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,state,TEXT,0,None,0
5,5,postal_code,TEXT,0,None,0
6,6,latitude,FLOAT,0,None,0
7,7,longitude,FLOAT,0,None,0
8,8,stars,FLOAT,0,None,0
9,9,review_count,BIGINT,0,None,0


In [19]:
with engine.begin() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS review")

# Load review data in chunks to avoid memory issues
for i, chunk in enumerate(
    pd.read_json(
        '../data/yelp_academic_dataset_review.json',
        lines=True,
        chunksize=50000
    )
):

    chunk.to_sql(
        'review',
        con=engine,
        if_exists='replace' if i == 0 else 'append',
        index=False
    )

    print(f"Loaded chunk {i+1}")

    if i == 9:   # 10 chunks × 50,000 = 500,000 reviews
        break

print("Review table loaded successfully")

Loaded chunk 1
Loaded chunk 2
Loaded chunk 3
Loaded chunk 4
Loaded chunk 5
Loaded chunk 6
Loaded chunk 7
Loaded chunk 8
Loaded chunk 9
Loaded chunk 10
Review table loaded successfully


In [20]:
pd.read_sql(
    "SELECT COUNT(*) AS review_count FROM review",
    engine
)

,review_count
0,500000


In [21]:
review_users = pd.read_sql(
    """
    SELECT DISTINCT user_id
    FROM review
    """,
    engine
)

review_users.shape

(318303, 1)

In [22]:
user_ids = set(review_users['user_id'])

print(len(user_ids))

318303


In [24]:
# Extract matching user records from the user JSON file
import json
import pandas as pd

matched_users = []

with open(
    '../data/yelp_academic_dataset_user.json',
    'r',
    encoding='utf-8'
) as f:

    for line in f:
        user = json.loads(line)

        if user['user_id'] in user_ids:
            matched_users.append(user)

user_df = pd.DataFrame(matched_users)

print(user_df.shape)

(318303, 22)


In [25]:
#Load user data into SQLite database
user_df.to_sql(
    'user',
    con=engine,
    if_exists='replace',
    index=False
)

print("User table loaded successfully")

User table loaded successfully


In [26]:
pd.read_sql(
    """
    SELECT COUNT(*) AS users
    FROM user
    """,
    engine
)

,users
0,318303


In [27]:
pd.read_sql(
    "PRAGMA table_info(business);",
    engine
)

,cid,name,type,notnull,dflt_value,pk
0,0,business_id,TEXT,0,None,0
1,1,name,TEXT,0,None,0
2,2,address,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,state,TEXT,0,None,0
5,5,postal_code,TEXT,0,None,0
6,6,latitude,FLOAT,0,None,0
7,7,longitude,FLOAT,0,None,0
8,8,stars,FLOAT,0,None,0
9,9,review_count,BIGINT,0,None,0


In [28]:
business = pd.read_sql(
    "SELECT * FROM business",
    engine
)

business.drop(
    ['attributes', 'hours'],
    axis=1,
    inplace=True
)

business.to_sql(
    'business',
    engine,
    if_exists='replace',
    index=False
)

150346